In [1]:
import pandas as pd

matches = pd.read_csv('../data/processed/matches_with_elo.csv')
matches['date'] = pd.to_datetime(matches['date'])
print(matches.shape)
print(matches.head(3))

(7429, 10)
        date home_team  away_team  home_score  away_score  \
0 1901-03-30   England   Scotland         2.0         2.0   
1 1902-05-03   England   Scotland         2.0         2.0   
2 1902-07-20   Uruguay  Argentina         0.0         6.0   

                  tournament  neutral  year  home_elo  away_elo  
0  British Home Championship    False  1901    2013.0    1973.0  
1  British Home Championship    False  1902    1995.0    1983.0  
2                   Friendly    False  1902    1879.0    2021.0  


In [3]:
matches['elo_diff'] = matches['home_elo']-matches['away_elo']
print(matches['elo_diff'].head())

0     40.0
1     12.0
2   -142.0
3     12.0
4    110.0
Name: elo_diff, dtype: float64


In [5]:
import numpy as np

conditions = [
    matches['home_score'] > matches['away_score'],
    matches['home_score'] < matches['away_score'],
    matches['home_score'] == matches['away_score']
]
choices = ['home_win', 'away_win', 'draw']

matches['result'] = np.select(conditions, choices, default='unknown')

In [6]:
print(matches[['home_score', 'away_score', 'result']].head(5))
print(matches['result'].value_counts())

   home_score  away_score    result
0         2.0         2.0      draw
1         2.0         2.0      draw
2         0.0         6.0  away_win
3         1.0         2.0  away_win
4         2.0         3.0  away_win
result
home_win    3442
away_win    2082
draw        1882
unknown       23
Name: count, dtype: int64


In [7]:
matches = matches[matches['result'] != 'unknown']
matches.to_csv('../data/processed/matches_features.csv', index=False)
print(matches.shape)

(7406, 12)


In [8]:
import pandas as pd
matches = pd.read_csv('../data/processed/matches_features.csv')
matches['date'] = pd.to_datetime(matches['date'])
print(matches.shape)
print(matches['neutral'].value_counts())

(7406, 12)
neutral
False    5479
True     1927
Name: count, dtype: int64


In [9]:
matches['is_neutral'] = matches['neutral'].astype(int)
print(matches[['neutral', 'is_neutral']].head(5))

   neutral  is_neutral
0    False           0
1    False           0
2    False           0
3    False           0
4    False           0


In [10]:
matches = matches.drop(columns=['neutral'])
print(matches.columns.tolist())

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'year', 'home_elo', 'away_elo', 'elo_diff', 'result', 'is_neutral']


In [29]:
def get_recent_form(team, date, df, n=5):
    matches_domicile = df['home_team'] == team
    matches_exterieur = df['away_team'] == team
    team_matches = df[(df['date'] < date) & (matches_domicile | matches_exterieur)]
    team_matches = team_matches.sort_values('date', ascending=False)
    recent = team_matches.head(n)
    
    if len(recent) == 0:
        return float('nan')
    
    home_wins = (recent['home_team'] == team) & (recent['result'] == 'home_win')
    away_wins = (recent['away_team'] == team) & (recent['result'] == 'away_win')
    wins = (home_wins | away_wins).sum()
    return wins / len(recent)

In [30]:
forme = get_recent_form("France", pd.Timestamp("2021-06-15"), matches)
print(forme)

0.8


In [31]:
import time
start = time.time()
get_recent_form("France", pd.Timestamp("2021-06-15"), matches)
print(f"Temps : {time.time() - start:.3f} secondes")

Temps : 0.006 secondes


In [32]:
matches['home_form'] = matches.apply(
    lambda row: get_recent_form(row['home_team'], row['date'], matches),
    axis=1
)
print(matches['home_form'].head(10))

0         NaN
1    0.000000
2         NaN
3    0.000000
4    1.000000
5    0.333333
6         NaN
7    0.000000
8    0.250000
9    0.000000
Name: home_form, dtype: float64


In [33]:
matches['away_form'] = matches.apply(
    lambda row: get_recent_form(row['away_team'], row['date'], matches),
    axis=1
)
print(matches['away_form'].head(10))

0     NaN
1    0.00
2     NaN
3    0.00
4    0.00
5    0.00
6     NaN
7     NaN
8    0.25
9     NaN
Name: away_form, dtype: float64


In [34]:
print(matches.columns.tolist())
print(matches.shape)
print(matches[['elo_diff', 'result', 'is_neutral', 'home_form', 'away_form']].head(5))

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'year', 'home_elo', 'away_elo', 'elo_diff', 'result', 'is_neutral', 'home_form', 'away_form']
(7406, 14)
   elo_diff    result  is_neutral  home_form  away_form
0      40.0      draw           0        NaN        NaN
1      12.0      draw           0        0.0        0.0
2    -142.0  away_win           0        NaN        NaN
3      12.0  away_win           0        0.0        0.0
4     110.0  away_win           0        1.0        0.0
